* This example tests factors using within-industry ranks and compares them with regression-residual and raw-factor versions. This section references *QEPM*, p. 117.
* The original example asked users to set `DB_URI` to a database. This version uses deterministic local data, so neither `DB_URI` nor a database is required.

# Parameter Settings
--------------

In [ ]:
import asyncio
import datetime as dt


In [ ]:
import numpy as np
import pyarrow as pa


# Example Factors
--------------------

I compare three approaches to reducing industry concentration:

* Rank the raw factor.
* Rank the raw factor within each industry.
* Regress the raw factor on industry dummy variables and rank the residuals.

## 1. Raw Factor Ranks
--------------------

In [ ]:
import pyarrow.compute as pc


In [ ]:
rng = np.random.default_rng(42)
n_dates = 6


Without industry adjustment, stocks with the highest values of the selected alpha factor `CFO2EV` are concentrated in banks and the broader financial sector.

## 2. Within-Industry Factor Ranks
----------------------

Adjusted Shenwan industry classifications are used as industry labels:

In [ ]:
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


Within-industry ranks produce a more balanced industry distribution.

## 3. Industry-Neutral Factors from Regression
--------------------------------

Another approach is linear regression with industries as dummy variables. The regression residual replaces the original factor to achieve industry neutrality:

In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


In [ ]:
from alphamind.portfolio.rankbuilder import rank_build
from alphamind.settlement.simplesettle import simple_settle


This method is less effective in the example. The adjustment is relatively small, and concentration in the broader financial sector remains.

# Backtest Results
------------------

I compare the three methods using a simple equal-weighted strategy that is long the top 20% of stocks and short the bottom 20%:

In [ ]:
labels = panel["industry"].to_pylist()
group_labels = [
    f"{trade_date.isoformat()}:{label}"
    for trade_date, label in zip(panel["trade_date"].to_pylist(), labels)
]


In [ ]:
groups = np.unique(group_labels, return_inverse=True)[1]
signal = panel["factor"].to_numpy(zero_copy_only=False)


In [ ]:
selected = rank_build(signal, use_rank=2, groups=groups).flatten().astype(float)


In [ ]:
weights = selected / 6.0
settlement = await asyncio.to_thread(
    simple_settle,
    weights,
    panel["dx"].to_numpy(zero_copy_only=False),
    groups=labels,
)


In [ ]:
assert selected.sum() == n_dates * 6
settlement
